# Weak-lensing convergence angular power spectrum

`Cell_CEmulator.get_Limber_Cells` computes dimensionless $C_\ell^{\kappa\kappa}$ with the Limber approximation. Provide **either** `z_s` (single source plane) **or** `dndz=(z, n)` (source distribution). `Pcb=False` uses total matter, including massive neutrinos.

This example uses sources within 0 < z <= 3 and `use_ccl=False`, so pyccl is not required. `non_linear='Emulator'`, `'halofit'`, and `'linear'` are supported here; this `'linear'` option is distinct from the removed `get_pknl(nltype='linear')`.

The implementation extrapolates P(k) outside its tabulated k range along the line of sight. These curves illustrate the API; high multipoles need a separate scale-cut and convergence assessment.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

project_root = Path.cwd()
if not (project_root / 'CEmulator').is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))


In [ ]:
from CEmulator.Emulator import Cell_CEmulator
emu = Cell_CEmulator(verbose=False)
emu.set_cosmos(Omegab=0.0486, Omegac=0.2604, H0=68.98,
               As=2.1e-9, ns=0.9665, mnu=0.06, checkbound=True)
ells = np.geomspace(20, 1000, 40)
cl_plane = emu.get_Limber_Cells(ells=ells, z_s=1.0, Pcb=False,
                                non_linear='Emulator', use_ccl=False)
cl_linear = emu.get_Limber_Cells(ells=ells, z_s=1.0, Pcb=False,
                                 non_linear='linear', use_ccl=False)

In [ ]:
z_source = np.linspace(0.01, 2.5, 200)
n_source = z_source**2 * np.exp(-(z_source / 0.7)**1.5)
n_source /= np.trapz(n_source, z_source)
cl_distribution = emu.get_Limber_Cells(
    ells=ells, dndz=(z_source, n_source), Pcb=False,
    non_linear='Emulator', use_ccl=False)
for values in (cl_plane, cl_linear, cl_distribution):
    assert values.shape == ells.shape
    assert np.all(np.isfinite(values)) and np.all(values > 0)

In [ ]:
prefactor = ells * (ells + 1) / (2 * np.pi)
plt.loglog(ells, prefactor * cl_plane, label='Source plane z_s=1, nonlinear')
plt.loglog(ells, prefactor * cl_linear, '--', label='Source plane z_s=1, linear')
plt.loglog(ells, prefactor * cl_distribution, label='Source distribution, nonlinear')
plt.xlabel(r'$\ell$')
plt.ylabel(r'$\ell(\ell+1) C_\ell^{\kappa\kappa}/(2\pi)$')
plt.legend()
plt.tight_layout()